In [1]:
import torch
import torch.nn.functional as F

In [30]:
def model_params_1ff(dm, n_laers, vs=50000):
    embedding = vs*dm
    block = (dm**2)*6
    ap = n_laers*block+2*embedding
    return ap, 2*embedding

def model_params_1ff_compressor(dm, dm_projected, n_laers=8, vs=50000):
    projection = dm*dm_projected
    embedding = projection
    block = projection*12
    ap = n_laers*block+3*embedding
    return ap, 3*embedding

def minitron_params_1ff(dm, n_laers=8, att_ratio=1.0, doner_dm=1024, vs=50000):
    embedding = vs*dm
    block_1 = (dm*dm)*2
    block_2 = (dm*doner_dm)*4
    block = block_1+block_2*att_ratio
    # block = (dm**2)*6
    ap = n_laers*block+2*embedding
    return ap, 2*embedding

def_t, def_e = model_params_1ff(768, 16)
min_t, min_e = minitron_params_1ff(768, 16)

def_t-def_e, min_t-min_e, 0.75*16
min_t, min_e, def_t, def_e

(146006016.0, 76800000, 133423104, 76800000)

In [ ]:
def model_params(dm, n_laers, dff=4, vs=50000):
    embedding = vs*dm
    att = (dm**2)*4
    ff = (dm*dm*dff)*2
    block = att+ff
    ap = n_laers*block+2*embedding
    return ap, 2*embedding

def pruned_model_params(dm, doner_dm, n_laers, dff=4, vs=50000):
    embedding = vs*dm
    att = (dm*doner_dm)*4
    ff = (dm*dm*dff)*2
    block = att+ff
    ap = n_laers*block+2*embedding
    return ap, 2*embedding

def compressor_active_parameters(dm, doner_dm, n_laers, dff=4, vs=50000):
    residual_model, res_emb = pruned_model_params(dm, doner_dm, n_laers, dff, vs)
    # frozen_model = model_params(doner_dm, n_laers, dff, vs) # dev frozen parameteres
    att_p = (dm*doner_dm)*4
    ff_p = (dm*doner_dm)*2 + (dm*dff*doner_dm*dff)*2
    projections_param = (att_p + ff_p) * n_laers + 2*(dm*doner_dm)
    # total_active = residual_model + projections_param
    total_active =  projections_param
    return total_active, (2*(dm*doner_dm))

def calculate_total_steps(parameters, tokens_step=512*512, tpp=20):
    return  (tpp*parameters)/(tokens_step)

DONER_DM = 1024*1.5
# DM = 768*1.5
DM = 1024
N_LAYERS = 16*1.5
DFF = 4

doner_ap, doner_emb = model_params(DONER_DM, N_LAYERS, DFF)
doner_s = calculate_total_steps(doner_ap)
pruned_ap, pruned_emb = pruned_model_params(DM, DONER_DM, N_LAYERS, DFF)
pruned_s = calculate_total_steps(pruned_ap)
compressor_ap, compressor_emb = compressor_active_parameters(DM, DONER_DM, N_LAYERS, DFF)
compressor_s = calculate_total_steps(compressor_ap)
compresor_total_s = calculate_total_steps(compressor_ap+doner_ap)

print(f"Doner dmodel: {DONER_DM}, prunned dm: {DM}, n_layters: {N_LAYERS}")
print(f"Doner model [{round(doner_ap/10**6, 2)}, {round(doner_emb/10**6, 2)}], opt. steps: {doner_s}")
print(f"Pruned model [{round(pruned_ap/10**6, 2)}, {round(pruned_emb/10**6, 2)}], opt. steps: {pruned_s}")
print(f"Compression ratio: {round(pruned_ap/doner_ap, 2)}")
print(f"Compressor model [{round(compressor_ap/10**6)}, {round(compressor_emb/10**6, 2)}], opt. steps: {compressor_s}")
print(f"Compressor model all params [{round((compressor_ap+doner_ap)/10**6, 2)}, {round((compressor_emb+doner_emb)/10**6, 2)}], opt. steps: {compresor_total_s}")


Doner dmodel: 1536.0, prunned dm: 1024, n_layters: 24.0
Doner model [833.08, 153.6], opt. steps: 63558.75
Pruned model [454.72, 102.4], opt. steps: 34692.5
Compression ratio: 0.55
Compressor model [1438, 3.15], opt. steps: 109680.0
Compressor model all params [2270.67, 156.75], opt. steps: 173238.75


In [141]:

def create_description(doner_dm, pruned_dm, n_layers, dff=4):
    doner_ap, doner_emb = model_params(doner_dm, n_layers, dff)
    doner_s = calculate_total_steps(doner_ap)
    pruned_ap, pruned_emb = pruned_model_params(pruned_dm, doner_dm, n_layers, dff)
    pruned_s = calculate_total_steps(pruned_ap)
    compressor_ap, compressor_emb = compressor_active_parameters(pruned_dm, doner_dm, n_layers, dff)
    compressor_s = calculate_total_steps(compressor_ap)
    compresor_total_s = calculate_total_steps(compressor_ap+doner_ap)

    print(f"Doner dmodel: {doner_dm}, prunned dm: {pruned_dm}, n_layters: {n_layers}")
    print(f"Doner model [{round(doner_ap/10**6, 2)}, {round(doner_emb/10**6, 2)}], opt. steps: {doner_s}")
    print(f"Pruned model [{round(pruned_ap/10**6, 2)}, {round(pruned_emb/10**6, 2)}], opt. steps: {pruned_s}")
    print(f"Compression ratio: {round(pruned_ap/doner_ap, 2)}")
    print(f"Compressor model [{round(compressor_ap/10**6)}, {round(compressor_emb/10**6, 2)}], opt. steps: {compressor_s}")
    print(f"Compressor model all params [{round((compressor_ap+doner_ap)/10**6, 2)}, {round((compressor_emb+doner_emb)/10**6, 2)}], opt. steps: {compresor_total_s}")

create_description(1536, 1024, 24, 4)

Doner dmodel: 1536, prunned dm: 1024, n_layters: 24
Doner model [833.08, 153.6], opt. steps: 63558.75
Pruned model [454.72, 102.4], opt. steps: 34692.5
Compression ratio: 0.55
Compressor model [1438, 3.15], opt. steps: 109680.0
Compressor model all params [2270.67, 156.75], opt. steps: 173238.75


In [ ]:
def compress_info(doner_dm, compressed_dm, n_layers):
    compressor_ap, compressor_ap_e = model_params_1ff_compressor(DM, DM_P, n_layers)
    default_ap, default_ap_e = model_params_1ff(compressed_dm, n_layers)
    doner_ap, doner_ap_e = model_params_1ff(doner_dm, n_layers)

    print(f"N layers: {n_layers}, doner dmodel: {doner_dm}, compressed dmodel: {compressed_dm}, head=layers: {compressed_dm/n_layers}")
    print("[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]")
    print(f"Compressor AP [M]: {round(compressor_ap/10**6, 2)} - {round(compressor_ap_e/10**6, 2)} = {round(compressor_ap/10**6 - compressor_ap_e/10**6, 2)}")
    print(f"Default AP [M]: {round(default_ap/10**6, 2)} - {round(default_ap_e/10**6, 2)} = {round(default_ap/10**6 - default_ap_e/10**6, 2)}")
    print(f"Doner AP [M]: {round(doner_ap/10**6, 2)} - {round(doner_ap_e/10**6, 2)} = {round(doner_ap/10**6 - doner_ap_e/10**6, 2)}")
    print(f"Compression ratio: {round(default_ap/doner_ap, 3)}; {round(doner_ap/10**6, 2)} -> {round(default_ap/10**6, 2)}")

In [ ]:
US = 20000
CR = 0.5
def calculate_compresor_flops_ratio(dm, doner_dm, n_layers, dff=4, vs=50000):
    compressor_ap, compressor_emb = compressor_active_parameters(dm, doner_dm, n_layers, dff, vs)
    doner_dm_fp, doner_dm_fp_emb = model_params(doner_dm, n_layers, dff, vs)
    pruned_ap, pruned_emb = pruned_model_params(dm, doner_dm, n_layers, dff, vs)
    
    compressor_ap = compressor_ap-compressor_emb
    doner_dm_fp = doner_dm_fp-doner_dm_fp_emb
    pruned_ap = pruned_ap-pruned_emb

    pruned_cost = pruned_ap*3
    # compressor_cost = pruned_ap + (pruned_ap+compressor_ap+doner_dm_fp)*2
    # compressor_cost = pruned_ap + (pruned_ap+compressor_ap)*2
    # compressor_cost = pruned_ap + (pruned_ap+compressor_ap)*2 + (dm*doner_dm*5 + (doner_dm*dm*dff*dff))*n_layers #  + (doner_dm*doner_dm)*4*n_layers
    # compressor_cost = (pruned_ap+compressor_ap+doner_dm_fp) + (pruned_ap+compressor_ap)*2

    compressor_cost = pruned_ap + (compressor_ap)*2 + (dm*doner_dm*5 + (doner_dm*dm*dff*dff))*n_layers
    
    return compressor_cost/pruned_cost, compressor_cost, pruned_cost

calculate_compresor_flops_ratio(DM, DONER_DM, N_LAYERS, DFF)# , calculate_compresor_flops_ratio(DONER_DM*CR, DONER_DM, N_LAYERS, DFF)#, calculate_compresor_flops_ratio(DM*US, DONER_DM*US, N_LAYERS*US, DFF)


(2.0476190476190474, 1623195648, 792723456)

In [40]:
for i in range(5):
    print(i*10179)

0
10179
20358
30537
40716


In [44]:
N_LAYERS = 24
DM = N_LAYERS*64

def compress_info(doner_dm, compressed_dm, n_layers):
    compressor_ap, compressor_ap_e = model_params_1ff_compressor(DM, DM_P, n_layers)
    default_ap, default_ap_e = model_params_1ff(compressed_dm, n_layers)
    doner_ap, doner_ap_e = model_params_1ff(doner_dm, n_layers)

    print(f"N layers: {n_layers}, doner dmodel: {doner_dm}, compressed dmodel: {compressed_dm}, head=layers: {compressed_dm/n_layers}")
    print("[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]")
    print(f"Compressor AP [M]: {round(compressor_ap/10**6, 2)} - {round(compressor_ap_e/10**6, 2)} = {round(compressor_ap/10**6 - compressor_ap_e/10**6, 2)}")
    print(f"Default AP [M]: {round(default_ap/10**6, 2)} - {round(default_ap_e/10**6, 2)} = {round(default_ap/10**6 - default_ap_e/10**6, 2)}")
    print(f"Doner AP [M]: {round(doner_ap/10**6, 2)} - {round(doner_ap_e/10**6, 2)} = {round(doner_ap/10**6 - doner_ap_e/10**6, 2)}")
    print(f"Compression ratio: {round(default_ap/doner_ap, 3)}; {round(doner_ap/10**6, 2)} -> {round(default_ap/10**6, 2)}")

print(f"Compression (layres: {N_LAYERS}) {DM} -> {N_LAYERS*52}")
compress_info(DM, N_LAYERS*52, N_LAYERS)
print()

print(f"Compression (layres: {N_LAYERS}) {DM} -> {N_LAYERS*48}")
compress_info(DM, N_LAYERS*48, N_LAYERS)
print()

print(f"Compression (layres: {N_LAYERS}) {DM} -> {N_LAYERS*42}")
compress_info(DM, N_LAYERS*42, N_LAYERS)
print()

print(f"Compression (layres: {N_LAYERS}) {DM} -> {N_LAYERS*32}")
compress_info(DM, N_LAYERS*32, N_LAYERS)
print()



Compression (layres: 24) 1536 -> 1248
N layers: 24, doner dmodel: 1536, compressed dmodel: 1248, head=layers: 52.0
[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]
Compressor AP [M]: 457.7 - 4.72 = 452.98
Default AP [M]: 349.08 - 124.8 = 224.28
Doner AP [M]: 493.34 - 153.6 = 339.74
Compression ratio: 0.708; 493.34 -> 349.08

Compression (layres: 24) 1536 -> 1152
N layers: 24, doner dmodel: 1536, compressed dmodel: 1152, head=layers: 48.0
[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]
Compressor AP [M]: 457.7 - 4.72 = 452.98
Default AP [M]: 306.3 - 115.2 = 191.1
Doner AP [M]: 493.34 - 153.6 = 339.74
Compression ratio: 0.621; 493.34 -> 306.3

Compression (layres: 24) 1536 -> 1008
N layers: 24, doner dmodel: 1536, compressed dmodel: 1008, head=layers: 42.0
[Model]: [Act. params] - [Embedding act. params] = [Non-enmbedding ap]
Compressor AP [M]: 457.7 - 4.72 = 452.98
Default AP [M]: 247.11 - 100.8 = 146.31
Doner AP [M]: 493.34 - 153.6 = 339.74


In [45]:
24*64

1536